# Static `env` schema analysis with AST

This notebook contains a self-contained implementation of `analyze_env_schema(function)` and validates it against all eight requested rules.

The analyzer is a path-sensitive symbolic interpreter. It never runs the analyzed function. It tracks the original `env` tree, independently tracked deep copies, active name aliases, overwrite state, usage state, direct root arguments, and logical execution paths.

### Deliberate semantics

- `branch = env["a"]` creates an alias and schema nodes but does not mark them used.
- Passing a tracked **tree root** directly to any function records a `FunctionCall` and does not mark that root used. This includes `copy.deepcopy(env)`, because it is also a direct root argument.
- Passing a branch, such as `consume(env["a"])`, is a normal use rather than a `FunctionCall`.
- Branch outputs are compared after remaining common statements. A conditional access can therefore converge when another path performs the same access later.
- Literal subscription keys are required. Dynamic keys raise `DynamicKeyError`.
- `DeepDiff` is used automatically when installed. A built-in structural diff keeps the notebook self-contained when it is absent.

## Implementation — copy this single cell into another notebook

In [33]:
from __future__ import annotations

import ast
import inspect
import json
import logging
import textwrap
from dataclasses import dataclass, field
from typing import Any, Hashable, Iterable, Literal, NamedTuple, Sequence, Dict

try:  # DeepDiff improves diagnostics, but the analyzer remains self-contained.
    from deepdiff import DeepDiff  # type: ignore
except ImportError:  # pragma: no cover - exercised when DeepDiff is not installed.
    DeepDiff = None  # type: ignore[assignment]


logger = logging.getLogger(__name__)


class SchemaAnalysisError(RuntimeError):
    """Base class for static env-schema analysis errors."""


class SourceUnavailableError(SchemaAnalysisError):
    """Raised when Python source cannot be recovered for a function handle."""


class DynamicKeyError(SchemaAnalysisError):
    """Raised when an env key cannot be determined statically."""


class PathExplosionError(SchemaAnalysisError):
    """Raised when control-flow expansion exceeds the configured path limit."""


class DynamicSchemaError(SchemaAnalysisError):
    """Raised when different execution paths produce different schemas.

    Attributes
    ----------
    branch_differences
        A list of dictionaries. Each dictionary identifies the baseline path,
        the compared path(s), and a structural diff between their outputs.
    """

    def __init__(self, branch_differences: list[dict[str, Any]]) -> None:
        self.branch_differences = branch_differences
        message = self._build_message(branch_differences)
        super().__init__(message)

    @staticmethod
    def _build_message(branch_differences: list[dict[str, Any]]) -> str:
        lines = [
            "Dynamic env schema detected: logical execution paths do not "
            "produce identical EnvTree/FunctionCall outputs."
        ]
        for index, item in enumerate(branch_differences, start=1):
            lines.append(f"\nDifference group {index}:")
            lines.append(
                "  baseline path(s): " + ", ".join(item["baseline_paths"])
            )
            lines.append(
                "  compared path(s): " + ", ".join(item["compared_paths"])
            )
            lines.append(
                textwrap.indent(
                    json.dumps(item["diff"], indent=2, default=repr), "  "
                )
            )
        return "\n".join(lines)


@dataclass(frozen=True)
class NodeRef:
    """Stable reference to a node in one symbolic EnvTree."""

    tree_id: int
    path: tuple[Hashable, ...] = ()


@dataclass(frozen=True)
class CopyOrigin:
    """Identifies the tree node from which a symbolic deep copy was made."""

    tree_id: int
    path: tuple[Hashable, ...]

    def to_dict(self) -> dict[str, Any]:
        return {
            "tree_id": self.tree_id,
            "path": [_encode_key(key) for key in self.path],
        }


@dataclass
class EnvNode:
    """One node in a symbolic dictionary tree.

    Parameters
    ----------
    key
        The dictionary key used to reach this node from its parent. The root
        stores a descriptive root key instead.
    used
        Whether this node has been used according to the analyzer's rules.
    overwritten
        Whether this node's value has been replaced. Once true, future reads
        cannot change ``used`` for this node or any overwritten descendant.
    parent
        Parent node, or ``None`` for a tree root.
    children
        Child nodes keyed by their literal dictionary keys.
    """

    key: Hashable
    used: bool = False
    overwritten: bool = False
    parent: EnvNode | None = field(default=None, repr=False, compare=False)
    children: dict[Hashable, EnvNode] = field(default_factory=dict)

    def ensure_child(self, key: Hashable) -> EnvNode:
        """Return an existing child, or create one with inherited overwrite state."""
        child = self.children.get(key)
        if child is None:
            child = EnvNode(
                key=key,
                used=False,
                overwritten=self.overwritten,
                parent=self,
            )
            self.children[key] = child
        return child

    def mark_used(self) -> None:
        """Mark this node and eligible existing descendants as used.

        Traversal stops at every overwritten node. Existing ``used=True``
        values are never cleared.
        """
        if self.overwritten:
            return
        self.used = True
        for child in self.children.values():
            child.mark_used()

    def mark_overwritten(self) -> None:
        """Mark this node and all existing descendants as overwritten."""
        self.overwritten = True
        for child in self.children.values():
            child.mark_overwritten()

    def clone(self, parent: EnvNode | None = None) -> EnvNode:
        """Deep-copy this node and its descendants while repairing parents."""
        cloned = EnvNode(
            key=self.key,
            used=self.used,
            overwritten=self.overwritten,
            parent=parent,
        )
        cloned.children = {
            key: child.clone(parent=cloned) for key, child in self.children.items()
        }
        return cloned

    def path(self) -> tuple[Hashable, ...]:
        """Return this node's path relative to its tree root."""
        keys: list[Hashable] = []
        current: EnvNode | None = self
        while current is not None and current.parent is not None:
            keys.append(current.key)
            current = current.parent
        keys.reverse()
        return tuple(keys)

    def to_dict(self) -> dict[str, Any]:
        """Return a deterministic, comparison-friendly representation."""
        ordered_children = sorted(
            self.children.values(), key=lambda node: _key_sort_token(node.key)
        )
        return {
            "key": _encode_key(self.key),
            "used": self.used,
            "overwritten": self.overwritten,
            "children": [child.to_dict() for child in ordered_children],
        }


@dataclass
class EnvTree:
    """A symbolic env dictionary or an independently tracked deep copy."""

    tree_id: int
    root: EnvNode
    copied_from: CopyOrigin | None = None
    reduced_to_overwrites: bool = False

    def get_node(self, path: Sequence[Hashable], *, create: bool = True) -> EnvNode:
        """Resolve a path relative to the root."""
        node = self.root
        for key in path:
            if create:
                node = node.ensure_child(key)
            else:
                try:
                    node = node.children[key]
                except KeyError as exc:
                    raise KeyError(tuple(path)) from exc
        return node

    def clone(self) -> EnvTree:
        """Deep-copy this complete tree."""
        return EnvTree(
            tree_id=self.tree_id,
            root=self.root.clone(),
            copied_from=self.copied_from,
            reduced_to_overwrites=self.reduced_to_overwrites,
        )

    def reduced_overwrite_copy(self) -> EnvTree:
        """Snapshot only ancestor paths leading to overwritten nodes.

        The root is always retained so the result remains a valid tree. If no
        overwritten node exists, the reduced tree consists of the root alone.
        """

        def prune(node: EnvNode, parent: EnvNode | None, *, keep_root: bool) -> EnvNode | None:
            kept_children: dict[Hashable, EnvNode] = {}
            placeholder = EnvNode(
                key=node.key,
                used=node.used,
                overwritten=node.overwritten,
                parent=parent,
            )
            for key, child in node.children.items():
                kept = prune(child, placeholder, keep_root=False)
                if kept is not None:
                    kept_children[key] = kept
            should_keep = keep_root or node.overwritten or bool(kept_children)
            if not should_keep:
                return None
            placeholder.children = kept_children
            return placeholder

        reduced_root = prune(self.root, None, keep_root=True)
        assert reduced_root is not None
        return EnvTree(
            tree_id=self.tree_id,
            root=reduced_root,
            copied_from=self.copied_from,
            reduced_to_overwrites=True,
        )

    def to_dict(self) -> dict[str, Any]:
        """Return a deterministic representation suitable for DeepDiff."""
        return {
            "tree_id": self.tree_id,
            "copied_from": (
                None if self.copied_from is None else self.copied_from.to_dict()
            ),
            "reduced_to_overwrites": self.reduced_to_overwrites,
            "root": self.root.to_dict(),
        }
    
    def get_used_leaves(self, env: Dict[str, Any]|None = None) -> set[str]:
        """Return dotted paths for all used leaf nodes in this tree. If env is provided, usage will be propagated to the leaves of the environment, reflecting the actual environment structure.

        The tree root is excluded from the returned paths, so a node reached as
        ``env["key1"]["key2"]`` is represented as ``"key1.key2"`` rather than
        ``"env.key1.key2"``.

        Returns
        -------
        set[str]
            Dotted paths to every ``EnvNode`` whose ``used`` attribute is ``True``.
            Paths are returned in the insertion order of the tree's children.
        """
        used_paths: set[str] = set()
        def add_leaves(deeper_env: Dict[str, Any], parent_path: tuple[str, ...]) -> None:
            for key, value in deeper_env.items():
                child_path = (*parent_path, str(key))
                if isinstance(value, dict):
                    add_leaves(value, child_path)
                else:
                    used_paths.add(".".join(child_path))
        def visit(node: EnvNode, env: Dict[str, Any]|None, parent_path: tuple[str, ...]) -> None:
            for child in node.children.values():
                child_path = (*parent_path, str(child.key))
                deeper_env = None
                if env is not None:
                    if not isinstance(env, dict):
                        raise ValueError(f"Environment path '{'.'.join(parent_path)}' is not a dictionary, thus it cannot contain '{child.key}'.")
                    deeper_env = env.get(str(child.key))
                    if deeper_env is None:
                        raise ValueError(f"'{child.key}' does not exist in environment path '{'.'.join(parent_path)}'.")
                if child.used and len(child.children) == 0:
                    if isinstance(deeper_env, dict):
                        add_leaves(deeper_env, child_path)
                    else:
                        used_paths.add(".".join(child_path))

                # Always inspect descendants. A child can be used even when its
                # parent is not, such as for env["branch"]["leaf"] when only the
                # final subscription counts as a use.
                visit(child, env=deeper_env, parent_path=child_path)

        visit(self.root, env=env, parent_path=())
        return used_paths
    
    def get_overwritten_leaves(self, env: Dict[str, Any]|None = None) -> set[str]:
        """Return dotted paths for all overwritten leaf nodes in this tree. If env is provided, overwrites will be propagated to the leaves of the environment, reflecting the actual environment structure.

        The tree root is excluded from the returned paths, so a node reached as
        ``env["key1"]["key2"]`` is represented as ``"key1.key2"`` rather than
        ``"env.key1.key2"``.

        Returns
        -------
        set[str]
            Dotted paths to every ``EnvNode`` whose ``overwritten`` attribute is ``True``.
            Paths are returned in the insertion order of the tree's children.
        """
        overwritten_paths: set[str] = set()
        def add_leaves(deeper_env: Dict[str, Any], parent_path: tuple[str, ...]) -> None:
            for key, value in deeper_env.items():
                child_path = (*parent_path, str(key))
                if isinstance(value, dict):
                    add_leaves(value, child_path)
                else:
                    overwritten_paths.add(".".join(child_path))
        def visit(node: EnvNode, env: Dict[str, Any]|None, parent_path: tuple[str, ...]) -> None:
            for child in node.children.values():
                child_path = (*parent_path, str(child.key))
                deeper_env = None
                if env is not None:
                    if not isinstance(env, dict):
                        raise ValueError(f"Environment path '{'.'.join(parent_path)}' is not a dictionary, thus it cannot contain '{child.key}'.")
                    deeper_env = env.get(str(child.key))
                    if deeper_env is None:
                        raise ValueError(f"'{child.key}' does not exist in environment path '{'.'.join(parent_path)}'.")
                if child.overwritten and len(child.children) == 0:
                    if isinstance(deeper_env, dict):
                        add_leaves(deeper_env, child_path)
                    else:
                        overwritten_paths.add(".".join(child_path))

                # Always inspect descendants. A child can be used even when its
                # parent is not, such as for env["branch"]["leaf"] when only the
                # final subscription counts as a use.
                visit(child, env=deeper_env, parent_path=child_path)

        visit(self.root, env=env, parent_path=())
        return overwritten_paths
        
    def __repr__(self) -> str:
        return json.dumps(self.to_dict(), indent=2, default=repr)

def get_qualified_name(local_name) -> str | None:
    local_path = local_name.split(".")
    obj = globals().get(local_path[0])
    if obj is None:
        return None
    for attr in local_path[1:]:
        obj = getattr(obj, attr, None)
        if obj is None:
            return None
    return f"{obj.__module__}.{obj.__qualname__}"

class FunctionCall(NamedTuple):
    """A function call receiving a tracked tree root.

    Fields
    ------
    function_name
        Static dotted/unparsed name of the called function.
    overwritten_tree
        Independent reduced snapshot containing only paths to overwritten
        nodes at the time the argument is evaluated.
    """

    function_name: str
    overwritten_tree: EnvTree

    def to_dict(self) -> dict[str, Any]:
        return {"function_name": self.function_name, "tree": self.overwritten_tree.to_dict()}

    def get_runtime_name(self) -> str:
        """Return the fully qualified name of the function at runtime."""
        runtime_name = get_qualified_name(self.function_name)
        if runtime_name is None:
            raise ValueError(f"Function call {self.function_name} not found in globals.")
        return runtime_name

    def get_overwritten_leaves(self, env: dict[str, Any] | None = None) -> set[str]:
        """Return the set of paths that are overwritten in this function call."""
        return self.overwritten_tree.get_overwritten_leaves(env=env)


@dataclass
class _AnalysisState:
    trees: list[EnvTree]
    active_aliases: dict[str, NodeRef]
    function_calls: list[FunctionCall]
    next_tree_id: int

    def clone(self) -> _AnalysisState:
        return _AnalysisState(
            trees=[tree.clone() for tree in self.trees],
            active_aliases=dict(self.active_aliases),
            function_calls=[
                FunctionCall(call.function_name, call.tree.clone())
                for call in self.function_calls
            ],
            next_tree_id=self.next_tree_id,
        )

    def tree(self, tree_id: int) -> EnvTree:
        try:
            return self.trees[tree_id]
        except (IndexError, TypeError) as exc:
            raise SchemaAnalysisError(f"Unknown symbolic tree id {tree_id}") from exc

    def node(self, ref: NodeRef, *, create: bool = True) -> EnvNode:
        return self.tree(ref.tree_id).get_node(ref.path, create=create)

    def snapshot(self) -> dict[str, Any]:
        """Return only the user-requested outputs, not transient aliases."""
        return {
            "trees": [tree.to_dict() for tree in self.trees],
            "function_calls": [call.to_dict() for call in self.function_calls],
        }


FlowStatus = Literal["normal", "return", "raise", "break", "continue"]


@dataclass
class _Flow:
    state: _AnalysisState
    status: FlowStatus = "normal"
    provenance: tuple[str, ...] = ("entry",)

    def branched(self, label: str) -> _Flow:
        return _Flow(
            state=self.state.clone(),
            status=self.status,
            provenance=(*self.provenance, label),
        )

    @property
    def label(self) -> str:
        return " -> ".join(self.provenance)


@dataclass(frozen=True)
class _ReferenceResult:
    ref: NodeRef
    pure_deepcopy_result: bool = False


class _EnvSchemaAnalyzer:
    """Path-sensitive symbolic interpreter for one function AST."""

    def __init__(
        self,
        function: Any,
        *,
        env_parameter: str,
        max_paths: int,
        deepcopy_names: Iterable[str],
    ) -> None:
        self.function = inspect.unwrap(function)
        self.env_parameter = env_parameter
        self.max_paths = max_paths
        self.deepcopy_names = frozenset(deepcopy_names)
        self.function_ast, self.source_start_line = self._extract_function_ast(
            self.function
        )
        self._validate_env_parameter()

    def analyze(self) -> tuple[list[EnvTree], list[FunctionCall]]:
        original_tree = EnvTree(
            tree_id=0,
            root=EnvNode(key=f"<root:{self.env_parameter}>"),
        )
        initial_state = _AnalysisState(
            trees=[original_tree],
            active_aliases={self.env_parameter: NodeRef(0, ())},
            function_calls=[],
            next_tree_id=1,
        )
        final_flows = self._exec_block(
            self.function_ast.body,
            [_Flow(state=initial_state)],
        )
        if not final_flows:
            final_flows = [_Flow(state=initial_state)]

        self._ensure_consistent_outputs(final_flows)
        canonical = final_flows[0].state
        return canonical.trees, canonical.function_calls

    @staticmethod
    def _extract_function_ast(function: Any) -> tuple[ast.FunctionDef | ast.AsyncFunctionDef, int]:
        try:
            source_lines, start_line = inspect.getsourcelines(function)
        except (OSError, IOError, TypeError) as exc:
            raise SourceUnavailableError(
                "Could not recover source for the supplied function handle. "
                "Define it in a .py file, or register notebook-generated source "
                "in linecache before analysis."
            ) from exc

        source = textwrap.dedent("".join(source_lines))
        try:
            module = ast.parse(source)
        except SyntaxError as exc:
            raise SourceUnavailableError("Recovered source could not be parsed.") from exc

        candidates = [
            node
            for node in ast.walk(module)
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == getattr(function, "__name__", None)
        ]
        if not candidates:
            raise SourceUnavailableError(
                f"Could not locate AST for function {getattr(function, '__name__', function)!r}."
            )
        # inspect.getsource normally places the requested definition first.
        candidates.sort(key=lambda node: (node.lineno, node.col_offset))
        return candidates[0], start_line

    def _validate_env_parameter(self) -> None:
        args = self.function_ast.args
        parameter_names = {
            arg.arg
            for arg in [*args.posonlyargs, *args.args, *args.kwonlyargs]
        }
        if args.vararg is not None:
            parameter_names.add(args.vararg.arg)
        if args.kwarg is not None:
            parameter_names.add(args.kwarg.arg)
        if self.env_parameter not in parameter_names:
            raise SchemaAnalysisError(
                f"Function {self.function_ast.name!r} has no parameter "
                f"named {self.env_parameter!r}."
            )

    def _line(self, node: ast.AST) -> int:
        return self.source_start_line + getattr(node, "lineno", 1) - 1

    def _check_path_count(self, flows: Sequence[_Flow]) -> None:
        if len(flows) > self.max_paths:
            raise PathExplosionError(
                f"Static control-flow expansion produced {len(flows)} paths, "
                f"exceeding max_paths={self.max_paths}."
            )

    def _exec_block(self, statements: Sequence[ast.stmt], flows: list[_Flow]) -> list[_Flow]:
        current = flows
        for statement in statements:
            next_flows: list[_Flow] = []
            for flow in current:
                if flow.status == "normal":
                    next_flows.extend(self._exec_stmt(statement, flow))
                else:
                    next_flows.append(flow)
            self._check_path_count(next_flows)
            current = next_flows
        return current

    def _exec_stmt(self, statement: ast.stmt, flow: _Flow) -> list[_Flow]:
        state = flow.state

        if isinstance(statement, ast.Assign):
            self._handle_assignment(statement.targets, statement.value, state)
            return [flow]

        if isinstance(statement, ast.AnnAssign):
            # A value-less local annotation does not rebind the existing name.
            if statement.value is not None:
                self._handle_assignment([statement.target], statement.value, state)
            if statement.annotation is not None:
                self._process_expr(statement.annotation, state)
            return [flow]

        if isinstance(statement, ast.AugAssign):
            # Per the requested rules, replacing a tracked node is an overwrite,
            # not a read, even though Python's runtime augmented assignment reads.
            self._handle_overwrite_target(statement.target, state)
            self._process_expr(statement.value, state)
            return [flow]

        if isinstance(statement, ast.Expr):
            self._process_expr(statement.value, state)
            return [flow]

        if isinstance(statement, ast.Delete):
            for target in statement.targets:
                self._handle_delete_target(target, state)
            return [flow]

        if isinstance(statement, ast.Return):
            self._process_expr(statement.value, state)
            flow.status = "return"
            return [flow]

        if isinstance(statement, ast.Raise):
            self._process_expr(statement.exc, state)
            self._process_expr(statement.cause, state)
            flow.status = "raise"
            return [flow]

        if isinstance(statement, ast.Assert):
            self._process_expr(statement.test, state)
            self._process_expr(statement.msg, state)
            return [flow]

        if isinstance(statement, ast.If):
            self._process_expr(statement.test, state)
            line = self._line(statement)
            body_flow = flow.branched(f"if@{line}:body")
            else_flow = flow.branched(f"if@{line}:else")
            body_outputs = self._exec_block(statement.body, [body_flow])
            else_outputs = self._exec_block(statement.orelse, [else_flow])
            return [*body_outputs, *else_outputs]

        if isinstance(statement, (ast.For, ast.AsyncFor)):
            return self._exec_for(statement, flow)

        if isinstance(statement, ast.While):
            return self._exec_while(statement, flow)

        if isinstance(statement, (ast.With, ast.AsyncWith)):
            for item in statement.items:
                self._process_expr(item.context_expr, state)
                if item.optional_vars is not None:
                    self._invalidate_target_aliases(item.optional_vars, state)
            return self._exec_block(statement.body, [flow])

        if isinstance(statement, ast.Match):
            return self._exec_match(statement, flow)

        if isinstance(statement, (ast.Try, getattr(ast, "TryStar", ast.Try))):
            return self._exec_try(statement, flow)

        if isinstance(statement, (ast.FunctionDef, ast.AsyncFunctionDef)):
            # A nested function body is not executed when defined. Decorators,
            # defaults, and annotations are evaluated now and therefore count.
            for decorator in statement.decorator_list:
                self._process_expr(decorator, state)
            for default in [*statement.args.defaults, *statement.args.kw_defaults]:
                self._process_expr(default, state)
            for arg in [
                *statement.args.posonlyargs,
                *statement.args.args,
                *statement.args.kwonlyargs,
            ]:
                self._process_expr(arg.annotation, state)
            self._process_expr(statement.returns, state)
            state.active_aliases.pop(statement.name, None)
            return [flow]

        if isinstance(statement, ast.ClassDef):
            for decorator in statement.decorator_list:
                self._process_expr(decorator, state)
            for base in statement.bases:
                self._process_expr(base, state)
            for keyword in statement.keywords:
                self._process_expr(keyword.value, state)
            state.active_aliases.pop(statement.name, None)
            return [flow]

        if isinstance(statement, (ast.Import, ast.ImportFrom)):
            for alias in statement.names:
                bound = alias.asname or alias.name.split(".", 1)[0]
                state.active_aliases.pop(bound, None)
            return [flow]

        if isinstance(statement, ast.Break):
            flow.status = "break"
            return [flow]

        if isinstance(statement, ast.Continue):
            flow.status = "continue"
            return [flow]

        if isinstance(statement, (ast.Pass, ast.Global, ast.Nonlocal)):
            return [flow]

        # Generic fallback: process executable expression children without
        # descending into nested statement blocks a second time.
        for child in ast.iter_child_nodes(statement):
            if isinstance(child, ast.expr):
                self._process_expr(child, state)
        return [flow]

    def _exec_for(self, statement: ast.For | ast.AsyncFor, flow: _Flow) -> list[_Flow]:
        state = flow.state
        self._process_expr(statement.iter, state)
        line = self._line(statement)

        zero = flow.branched(f"for@{line}:zero-iterations")
        one = flow.branched(f"for@{line}:one-or-more-iterations")
        self._invalidate_target_aliases(statement.target, one.state)

        zero_outputs = self._exec_block(statement.orelse, [zero])
        one_outputs = self._exec_block(statement.body, [one])

        normalized: list[_Flow] = []
        for candidate in one_outputs:
            if candidate.status == "break":
                candidate.status = "normal"
                candidate.provenance = (*candidate.provenance, "loop-break")
                normalized.append(candidate)
            elif candidate.status == "continue":
                candidate.status = "normal"
                candidate.provenance = (*candidate.provenance, "loop-continue/exhaust")
                normalized.extend(self._exec_block(statement.orelse, [candidate]))
            elif candidate.status == "normal":
                normalized.extend(self._exec_block(statement.orelse, [candidate]))
            else:
                normalized.append(candidate)
        return [*zero_outputs, *normalized]

    def _exec_while(self, statement: ast.While, flow: _Flow) -> list[_Flow]:
        state = flow.state
        self._process_expr(statement.test, state)
        line = self._line(statement)

        zero = flow.branched(f"while@{line}:zero-iterations")
        one = flow.branched(f"while@{line}:one-or-more-iterations")
        zero_outputs = self._exec_block(statement.orelse, [zero])
        one_outputs = self._exec_block(statement.body, [one])

        normalized: list[_Flow] = []
        for candidate in one_outputs:
            if candidate.status == "break":
                candidate.status = "normal"
                candidate.provenance = (*candidate.provenance, "loop-break")
                normalized.append(candidate)
            elif candidate.status == "continue":
                candidate.status = "normal"
                candidate.provenance = (*candidate.provenance, "loop-continue/exhaust")
                normalized.extend(self._exec_block(statement.orelse, [candidate]))
            elif candidate.status == "normal":
                normalized.extend(self._exec_block(statement.orelse, [candidate]))
            else:
                normalized.append(candidate)
        return [*zero_outputs, *normalized]

    def _exec_match(self, statement: ast.Match, flow: _Flow) -> list[_Flow]:
        self._process_expr(statement.subject, flow.state)
        line = self._line(statement)
        outputs: list[_Flow] = []

        for index, case in enumerate(statement.cases):
            case_flow = flow.branched(f"match@{line}:case-{index}")
            self._invalidate_pattern_bindings(case.pattern, case_flow.state)
            self._process_pattern_values(case.pattern, case_flow.state)
            self._process_expr(case.guard, case_flow.state)
            outputs.extend(self._exec_block(case.body, [case_flow]))

        if not self._match_is_exhaustive(statement.cases):
            outputs.append(flow.branched(f"match@{line}:no-match"))
        return outputs

    def _exec_try(self, statement: ast.Try, flow: _Flow) -> list[_Flow]:
        line = self._line(statement)

        success = flow.branched(f"try@{line}:body-success")
        success_outputs = self._exec_block(statement.body, [success])
        with_else: list[_Flow] = []
        for candidate in success_outputs:
            if candidate.status == "normal":
                with_else.extend(self._exec_block(statement.orelse, [candidate]))
            else:
                with_else.append(candidate)

        handler_outputs: list[_Flow] = []
        for index, handler in enumerate(statement.handlers):
            handler_flow = flow.branched(f"try@{line}:except-{index}")
            self._process_expr(handler.type, handler_flow.state)
            if handler.name:
                handler_flow.state.active_aliases.pop(handler.name, None)
            handler_outputs.extend(self._exec_block(handler.body, [handler_flow]))

        alternatives = [*with_else, *handler_outputs]
        if not statement.handlers:
            alternatives = with_else

        if statement.finalbody:
            alternatives = self._apply_finally(statement.finalbody, alternatives)
        return alternatives

    def _apply_finally(
        self, finalbody: Sequence[ast.stmt], flows: Sequence[_Flow]
    ) -> list[_Flow]:
        outputs: list[_Flow] = []
        for flow in flows:
            incoming_status = flow.status
            temporary = _Flow(
                state=flow.state,
                status="normal",
                provenance=(*flow.provenance, "finally"),
            )
            final_outputs = self._exec_block(finalbody, [temporary])
            for candidate in final_outputs:
                if candidate.status == "normal":
                    candidate.status = incoming_status
                outputs.append(candidate)
        return outputs

    def _handle_assignment(
        self,
        targets: Sequence[ast.expr],
        value: ast.expr,
        state: _AnalysisState,
    ) -> None:
        # Python evaluates the RHS before rebinding name targets. Resolve it
        # first so expressions such as ``alias = alias + 1`` still see the old
        # active alias. Exact tracked references are classified without usage.
        reference = self._eval_reference(value, state)
        all_simple_names = all(isinstance(target, ast.Name) for target in targets)

        if all_simple_names:
            if reference is None:
                self._process_expr(value, state)
                for target in targets:
                    assert isinstance(target, ast.Name)
                    state.active_aliases.pop(target.id, None)
            else:
                # Pure name-to-node assignment is alias creation and never use.
                for target in targets:
                    assert isinstance(target, ast.Name)
                    state.active_aliases[target.id] = reference.ref
            return

        # Pre-mark only dictionary slots that are already known aliases. This
        # suppresses reads of the overwritten node on the same line without
        # prematurely rebinding names in tuple/list targets before the RHS.
        premarked_targets: set[int] = set()
        for target in targets:
            if not isinstance(target, ast.Name):
                self._premark_assignment_overwrites(
                    target, state, premarked_targets
                )

        if reference is None:
            self._process_expr(value, state)
        elif not reference.pure_deepcopy_result:
            state.node(reference.ref).mark_used()

        # Python applies assignment targets after evaluating the RHS. Direct
        # name targets can become aliases; names nested in unpacking targets
        # cannot be assumed to receive the complete tracked value.
        for target in targets:
            if isinstance(target, ast.Name):
                if reference is None:
                    state.active_aliases.pop(target.id, None)
                else:
                    state.active_aliases[target.id] = reference.ref
            else:
                self._finish_assignment_target(
                    target, state, premarked_targets
                )

    def _premark_assignment_overwrites(
        self,
        target: ast.expr,
        state: _AnalysisState,
        premarked_targets: set[int],
    ) -> None:
        """Mark tracked slot targets before RHS usage analysis."""
        if isinstance(target, (ast.Tuple, ast.List)):
            for element in target.elts:
                self._premark_assignment_overwrites(
                    element, state, premarked_targets
                )
            return
        if isinstance(target, ast.Starred):
            self._premark_assignment_overwrites(
                target.value, state, premarked_targets
            )
            return
        if isinstance(target, ast.Subscript):
            reference = self._eval_reference(target, state)
            if reference is not None:
                state.node(reference.ref).mark_overwritten()
                premarked_targets.add(id(target))

    def _finish_assignment_target(
        self,
        target: ast.expr,
        state: _AnalysisState,
        premarked_targets: set[int],
    ) -> None:
        """Apply a non-name assignment target after RHS evaluation."""
        if isinstance(target, ast.Name):
            state.active_aliases.pop(target.id, None)
            return
        if isinstance(target, (ast.Tuple, ast.List)):
            for element in target.elts:
                self._finish_assignment_target(
                    element, state, premarked_targets
                )
            return
        if isinstance(target, ast.Starred):
            self._finish_assignment_target(
                target.value, state, premarked_targets
            )
            return
        if isinstance(target, ast.Subscript):
            if id(target) in premarked_targets:
                return
            reference = self._eval_reference(target, state)
            if reference is not None:
                state.node(reference.ref).mark_overwritten()
            else:
                self._process_expr(target.value, state)
                self._process_expr(target.slice, state)
            return
        if isinstance(target, ast.Attribute):
            self._process_expr(target.value, state)
            return
        self._invalidate_target_aliases(target, state)

    def _handle_overwrite_target(self, target: ast.expr, state: _AnalysisState) -> None:
        if isinstance(target, ast.Name):
            # Rebinding a Python variable overrides an alias; it does not mutate
            # the symbolic dictionary node to which the alias used to point.
            state.active_aliases.pop(target.id, None)
            return

        if isinstance(target, (ast.Tuple, ast.List)):
            for element in target.elts:
                self._handle_overwrite_target(element, state)
            return

        if isinstance(target, ast.Starred):
            self._handle_overwrite_target(target.value, state)
            return

        if isinstance(target, ast.Subscript):
            reference = self._eval_reference(target, state)
            if reference is not None:
                state.node(reference.ref).mark_overwritten()
                return
            # Untracked container assignment can still use env in its base/key.
            self._process_expr(target.value, state)
            self._process_expr(target.slice, state)
            return

        if isinstance(target, ast.Attribute):
            # Assigning an attribute mutates the object stored in a node, not
            # the dictionary slot itself, so the base expression is a usage.
            self._process_expr(target.value, state)
            return

        self._invalidate_target_aliases(target, state)

    def _handle_delete_target(self, target: ast.expr, state: _AnalysisState) -> None:
        if isinstance(target, ast.Name):
            state.active_aliases.pop(target.id, None)
            return
        if isinstance(target, (ast.Tuple, ast.List)):
            for element in target.elts:
                self._handle_delete_target(element, state)
            return
        if isinstance(target, ast.Subscript):
            reference = self._eval_reference(target, state)
            if reference is not None:
                state.node(reference.ref).mark_overwritten()
                return
        self._handle_overwrite_target(target, state)

    def _invalidate_target_aliases(self, target: ast.AST, state: _AnalysisState) -> None:
        if isinstance(target, ast.Name):
            state.active_aliases.pop(target.id, None)
        elif isinstance(target, (ast.Tuple, ast.List)):
            for element in target.elts:
                self._invalidate_target_aliases(element, state)
        elif isinstance(target, ast.Starred):
            self._invalidate_target_aliases(target.value, state)
        elif isinstance(target, ast.Subscript):
            self._handle_overwrite_target(target, state)
        elif isinstance(target, ast.Attribute):
            self._process_expr(target.value, state)

    def _process_expr(self, expression: ast.AST | None, state: _AnalysisState) -> None:
        if expression is None:
            return

        if isinstance(expression, ast.NamedExpr):
            reference = self._eval_reference(expression.value, state)
            if isinstance(expression.target, ast.Name):
                if reference is None:
                    # The old alias remains visible while the value is evaluated.
                    self._process_expr(expression.value, state)
                    state.active_aliases.pop(expression.target.id, None)
                else:
                    state.active_aliases[expression.target.id] = reference.ref
            else:
                self._handle_overwrite_target(expression.target, state)
                if reference is None:
                    self._process_expr(expression.value, state)
                elif not reference.pure_deepcopy_result:
                    state.node(reference.ref).mark_used()
            return

        reference = self._eval_reference(expression, state)
        if reference is not None:
            if not reference.pure_deepcopy_result:
                state.node(reference.ref).mark_used()
            return

        if isinstance(expression, ast.Call):
            self._process_call(expression, state)
            return

        if isinstance(expression, ast.Lambda):
            # Merely creating a lambda does not execute its body.
            return

        if isinstance(expression, ast.IfExp):
            # Expression-level branching is analyzed conservatively in place.
            # The statement-level path engine handles the control-flow forms
            # explicitly required by the API.
            self._process_expr(expression.test, state)
            left = state.clone()
            right = state.clone()
            self._process_expr(expression.body, left)
            self._process_expr(expression.orelse, right)
            self._merge_expression_branches(state, left, right, expression)
            return

        if isinstance(expression, ast.BoolOp):
            # Every operand is a possible evaluated operand. Analyze each so no
            # access is silently omitted; statement-level branching remains the
            # source of DynamicSchemaError path diagnostics.
            for value in expression.values:
                self._process_expr(value, state)
            return

        if isinstance(expression, (ast.ListComp, ast.SetComp, ast.GeneratorExp)):
            self._process_comprehension(expression, state)
            return

        if isinstance(expression, ast.DictComp):
            for generator in expression.generators:
                self._process_expr(generator.iter, state)
                for condition in generator.ifs:
                    self._process_expr(condition, state)
            self._process_expr(expression.key, state)
            self._process_expr(expression.value, state)
            return

        # For regular expressions, recursively analyze expression children. A
        # nested maximal env reference is consumed by the recursive call, so a
        # chain such as env['a']['b'] marks only the final node, not 'a'.
        for child in ast.iter_child_nodes(expression):
            if isinstance(child, ast.expr):
                self._process_expr(child, state)
            elif isinstance(child, ast.comprehension):
                self._process_expr(child.iter, state)
                for condition in child.ifs:
                    self._process_expr(condition, state)

    def _process_comprehension(
        self,
        expression: ast.ListComp | ast.SetComp | ast.GeneratorExp,
        state: _AnalysisState,
    ) -> None:
        for generator in expression.generators:
            self._process_expr(generator.iter, state)
            for condition in generator.ifs:
                self._process_expr(condition, state)
        self._process_expr(expression.elt, state)

    def _merge_expression_branches(
        self,
        destination: _AnalysisState,
        left: _AnalysisState,
        right: _AnalysisState,
        expression: ast.IfExp,
    ) -> None:
        left_snapshot = left.snapshot()
        right_snapshot = right.snapshot()
        if left_snapshot != right_snapshot:
            diff = _make_diff(left_snapshot, right_snapshot)
            line = self._line(expression)
            details = [
                {
                    "baseline_paths": [f"if-expression@{line}:body"],
                    "compared_paths": [f"if-expression@{line}:else"],
                    "diff": diff,
                }
            ]
            error = DynamicSchemaError(details)
            logger.error("%s", error)
            raise error
        replacement = left.clone()
        destination.trees = replacement.trees
        destination.active_aliases = replacement.active_aliases
        destination.function_calls = replacement.function_calls
        destination.next_tree_id = replacement.next_tree_id

    def _process_call(self, call: ast.Call, state: _AnalysisState) -> None:
        function_name = _call_name(call.func)

        # The callable expression itself may use a tracked node, e.g.
        # env['factory'](...). It is not a root argument.
        self._process_expr(call.func, state)

        for argument in call.args:
            if isinstance(argument, ast.Starred):
                self._process_expr(argument.value, state)
                continue
            self._process_call_argument(argument, function_name, state)

        for keyword in call.keywords:
            if keyword.arg is None:  # **mapping
                self._process_expr(keyword.value, state)
            else:
                self._process_call_argument(keyword.value, function_name, state)

    def _process_call_argument(
        self,
        argument: ast.expr,
        function_name: str,
        state: _AnalysisState,
    ) -> None:
        reference = self._eval_reference(argument, state)
        if reference is None:
            self._process_expr(argument, state)
            return

        if reference.ref.path == ():
            tree = state.tree(reference.ref.tree_id)
            state.function_calls.append(
                FunctionCall(function_name, tree.reduced_overwrite_copy())
            )
            return

        # Only roots receive the special FunctionCall treatment. Passing a branch
        # is a normal use of that branch.
        if not reference.pure_deepcopy_result:
            state.node(reference.ref).mark_used()

    def _eval_reference(
        self, expression: ast.AST, state: _AnalysisState
    ) -> _ReferenceResult | None:
        """Resolve an expression that evaluates exactly to a tracked node.

        This method builds missing schema nodes but never marks usage. It may
        create a new independent EnvTree when the expression is a recognized
        ``deepcopy`` call.
        """
        if isinstance(expression, ast.Name):
            ref = state.active_aliases.get(expression.id)
            return None if ref is None else _ReferenceResult(ref)

        if isinstance(expression, ast.NamedExpr) and isinstance(
            expression.target, ast.Name
        ):
            value = self._eval_reference(expression.value, state)
            if value is None:
                return None
            state.active_aliases[expression.target.id] = value.ref
            return value

        if isinstance(expression, ast.Subscript):
            base = self._eval_reference(expression.value, state)
            if base is None:
                return None
            key = _literal_key(expression.slice, line=self._line(expression))
            parent = state.node(base.ref)
            parent.ensure_child(key)
            return _ReferenceResult(
                NodeRef(base.ref.tree_id, (*base.ref.path, key)),
                pure_deepcopy_result=False,
            )

        if isinstance(expression, ast.Call):
            if self._is_deepcopy_call(expression):
                if not expression.args:
                    return None
                source = self._eval_reference(expression.args[0], state)
                if source is None:
                    return None
                # Additional arguments are still direct arguments to deepcopy,
                # so apply the ordinary root-pass rule to each of them.
                deepcopy_name = _call_name(expression.func)
                for extra in expression.args[1:]:
                    if isinstance(extra, ast.Starred):
                        self._process_expr(extra.value, state)
                    else:
                        self._process_call_argument(extra, deepcopy_name, state)
                for keyword in expression.keywords:
                    if keyword.arg is None:
                        self._process_expr(keyword.value, state)
                    else:
                        self._process_call_argument(
                            keyword.value, deepcopy_name, state
                        )
                # deepcopy is itself a function call. Under the requested rule,
                # passing a tracked root records a FunctionCall in addition to
                # creating the independent copied tree. Deepcopying a branch is
                # not a root pass and therefore creates no FunctionCall.
                if source.ref.path == ():
                    source_tree = state.tree(source.ref.tree_id)
                    state.function_calls.append(
                        FunctionCall(
                            deepcopy_name,
                            source_tree.reduced_overwrite_copy(),
                        )
                    )
                new_ref = self._create_deepcopy(source.ref, state)
                return _ReferenceResult(new_ref, pure_deepcopy_result=True)

            if self._is_tracked_get_call(expression):
                attribute = expression.func
                assert isinstance(attribute, ast.Attribute)
                base = self._eval_reference(attribute.value, state)
                if base is None:
                    return None
                key = _literal_key(expression.args[0], line=self._line(expression))
                state.node(base.ref).ensure_child(key)
                return _ReferenceResult(
                    NodeRef(base.ref.tree_id, (*base.ref.path, key)),
                    pure_deepcopy_result=False,
                )

        return None

    def _create_deepcopy(self, source_ref: NodeRef, state: _AnalysisState) -> NodeRef:
        source_node = state.node(source_ref)
        new_id = state.next_tree_id
        state.next_tree_id += 1
        new_tree = EnvTree(
            tree_id=new_id,
            root=source_node.clone(parent=None),
            copied_from=CopyOrigin(source_ref.tree_id, source_ref.path),
        )
        # IDs are deliberately contiguous so list index and tree_id coincide.
        if new_id != len(state.trees):
            raise SchemaAnalysisError(
                "Internal tree-id invariant failed while creating deepcopy."
            )
        state.trees.append(new_tree)
        return NodeRef(new_id, ())

    def _is_deepcopy_call(self, call: ast.Call) -> bool:
        return _call_name(call.func) in self.deepcopy_names

    def _is_tracked_get_call(self, call: ast.Call) -> bool:
        return (
            isinstance(call.func, ast.Attribute)
            and call.func.attr == "get"
            and len(call.args) == 1
            and not call.keywords
            and self._eval_reference_without_side_effects(call.func.value) is not None
        )

    def _eval_reference_without_side_effects(self, expression: ast.AST) -> bool | None:
        """Syntactic precheck used only to recognize potential tracked .get calls."""
        if isinstance(expression, ast.Name):
            return True
        if isinstance(expression, ast.Subscript):
            return self._eval_reference_without_side_effects(expression.value)
        if isinstance(expression, ast.Call) and self._is_deepcopy_call(expression):
            return True
        return None

    def _invalidate_pattern_bindings(
        self, pattern: ast.pattern, state: _AnalysisState
    ) -> None:
        for name in _pattern_bound_names(pattern):
            state.active_aliases.pop(name, None)

    def _process_pattern_values(
        self, pattern: ast.pattern, state: _AnalysisState
    ) -> None:
        if isinstance(pattern, ast.MatchValue):
            self._process_expr(pattern.value, state)
        elif isinstance(pattern, ast.MatchClass):
            self._process_expr(pattern.cls, state)
            for child in [*pattern.patterns, *pattern.kwd_patterns]:
                self._process_pattern_values(child, state)
        elif isinstance(pattern, ast.MatchMapping):
            for key in pattern.keys:
                self._process_expr(key, state)
            for child in pattern.patterns:
                self._process_pattern_values(child, state)
        elif isinstance(pattern, ast.MatchSequence):
            for child in pattern.patterns:
                self._process_pattern_values(child, state)
        elif isinstance(pattern, ast.MatchOr):
            for child in pattern.patterns:
                self._process_pattern_values(child, state)
        elif isinstance(pattern, ast.MatchAs) and pattern.pattern is not None:
            self._process_pattern_values(pattern.pattern, state)

    @staticmethod
    def _match_is_exhaustive(cases: Sequence[ast.match_case]) -> bool:
        if not cases:
            return False
        last = cases[-1]
        if last.guard is not None:
            return False
        pattern = last.pattern
        # ``case _`` and an unguarded capture pattern are irrefutable.
        return isinstance(pattern, ast.MatchAs) and pattern.pattern is None

    def _ensure_consistent_outputs(self, flows: Sequence[_Flow]) -> None:
        groups: list[dict[str, Any]] = []
        for flow in flows:
            snapshot = flow.state.snapshot()
            matching = next(
                (group for group in groups if group["snapshot"] == snapshot), None
            )
            if matching is None:
                groups.append({"snapshot": snapshot, "paths": [flow.label]})
            else:
                matching["paths"].append(flow.label)

        if len(groups) <= 1:
            return

        baseline = groups[0]
        differences: list[dict[str, Any]] = []
        for group in groups[1:]:
            differences.append(
                {
                    "baseline_paths": baseline["paths"],
                    "compared_paths": group["paths"],
                    "diff": _make_diff(baseline["snapshot"], group["snapshot"]),
                }
            )

        error = DynamicSchemaError(differences)
        logger.error("%s", error)
        raise error


def analyze_env_schema(
    function: Any,
    *,
    env_parameter: str = "env",
    max_paths: int = 256,
    deepcopy_names: Iterable[str] = ("copy.deepcopy", "deepcopy"),
) -> tuple[list[EnvTree], list[FunctionCall]]:
    """Statically analyze how a function accesses an env dictionary tree.

    Parameters
    ----------
    function
        Function or unbound-method handle whose source can be recovered with
        :mod:`inspect`.
    env_parameter
        Name of the parameter representing the root dictionary.
    max_paths
        Maximum number of symbolic execution paths allowed after expanding
        control flow.
    deepcopy_names
        Static call names treated as deep-copy operations. The defaults support
        both ``copy.deepcopy(env)`` and ``from copy import deepcopy``.

    Returns
    -------
    trees, function_calls
        ``trees`` contains the original symbolic env plus every independently
        tracked deep copy. ``function_calls`` contains snapshots for every direct
        function argument that is a tracked tree root.

    Raises
    ------
    DynamicSchemaError
        If different logical execution paths finish with different tree/call
        outputs.
    DynamicKeyError
        If a tracked dictionary access uses a non-literal key.
    SourceUnavailableError
        If source code cannot be recovered from the function handle.

    Notes
    -----
    The analyzer recognizes literal subscription keys and ``tracked.get(key)``
    with exactly one literal argument. It treats direct root arguments specially
    as requested; branches passed to functions are ordinary uses. Loops are
    conservatively expanded as zero iterations versus one-or-more iterations.
    """
    analyzer = _EnvSchemaAnalyzer(
        function,
        env_parameter=env_parameter,
        max_paths=max_paths,
        deepcopy_names=deepcopy_names,
    )
    return analyzer.analyze()


def analysis_to_dict(
    trees: Sequence[EnvTree], calls: Sequence[FunctionCall]
) -> dict[str, Any]:
    """Convert analyzer output to plain deterministic dictionaries."""
    return {
        "trees": [tree.to_dict() for tree in trees],
        "function_calls": [call.to_dict() for call in calls],
    }


def print_analysis(trees: Sequence[EnvTree], calls: Sequence[FunctionCall]) -> None:
    """Pretty-print analyzer output."""
    print(json.dumps(analysis_to_dict(trees, calls), indent=2, default=repr))


def _literal_key(node: ast.AST, *, line: int) -> Hashable:
    try:
        key = ast.literal_eval(node)
    except (ValueError, TypeError, SyntaxError) as exc:
        expression = ast.unparse(node) if hasattr(ast, "unparse") else ast.dump(node)
        raise DynamicKeyError(
            f"Env key at line {line} is not statically literal: {expression}"
        ) from exc
    try:
        hash(key)
    except TypeError as exc:
        raise DynamicKeyError(
            f"Env key at line {line} is not hashable: {key!r}"
        ) from exc
    return key


def _call_name(function: ast.expr) -> str:
    if isinstance(function, ast.Name):
        return function.id
    if isinstance(function, ast.Attribute):
        parts: list[str] = []
        current: ast.AST = function
        while isinstance(current, ast.Attribute):
            parts.append(current.attr)
            current = current.value
        if isinstance(current, ast.Name):
            parts.append(current.id)
            return ".".join(reversed(parts))
    try:
        return ast.unparse(function)
    except Exception:  # pragma: no cover - ast.unparse is available on 3.9+.
        return ast.dump(function, include_attributes=False)


def _pattern_bound_names(pattern: ast.pattern) -> set[str]:
    names: set[str] = set()
    if isinstance(pattern, ast.MatchAs):
        if pattern.name is not None:
            names.add(pattern.name)
        if pattern.pattern is not None:
            names.update(_pattern_bound_names(pattern.pattern))
    elif isinstance(pattern, ast.MatchStar):
        if pattern.name is not None:
            names.add(pattern.name)
    elif isinstance(pattern, ast.MatchMapping):
        if pattern.rest is not None:
            names.add(pattern.rest)
        for child in pattern.patterns:
            names.update(_pattern_bound_names(child))
    elif isinstance(pattern, ast.MatchSequence):
        for child in pattern.patterns:
            names.update(_pattern_bound_names(child))
    elif isinstance(pattern, ast.MatchClass):
        for child in [*pattern.patterns, *pattern.kwd_patterns]:
            names.update(_pattern_bound_names(child))
    elif isinstance(pattern, ast.MatchOr):
        for child in pattern.patterns:
            names.update(_pattern_bound_names(child))
    return names


def _key_sort_token(key: Hashable) -> tuple[str, str]:
    return type(key).__name__, repr(key)


def _encode_key(key: Hashable) -> dict[str, str]:
    return {"type": type(key).__name__, "repr": repr(key)}


def _make_diff(left: Any, right: Any) -> Any:
    if DeepDiff is not None:
        diff = DeepDiff(left, right, ignore_order=False, verbose_level=2)
        try:
            return diff.to_dict()
        except AttributeError:  # pragma: no cover - compatibility fallback.
            return dict(diff)
    return {"fallback_structural_diff": _structural_diff(left, right)}


def _structural_diff(left: Any, right: Any, path: str = "root") -> list[dict[str, Any]]:
    differences: list[dict[str, Any]] = []
    if type(left) is not type(right):
        return [
            {
                "path": path,
                "kind": "type_changed",
                "left": type(left).__name__,
                "right": type(right).__name__,
            }
        ]

    if isinstance(left, dict):
        left_keys = set(left)
        right_keys = set(right)
        for key in sorted(left_keys - right_keys, key=repr):
            differences.append(
                {"path": f"{path}[{key!r}]", "kind": "removed", "left": left[key]}
            )
        for key in sorted(right_keys - left_keys, key=repr):
            differences.append(
                {"path": f"{path}[{key!r}]", "kind": "added", "right": right[key]}
            )
        for key in sorted(left_keys & right_keys, key=repr):
            differences.extend(
                _structural_diff(left[key], right[key], f"{path}[{key!r}]")
            )
        return differences

    if isinstance(left, list):
        common = min(len(left), len(right))
        for index in range(common):
            differences.extend(
                _structural_diff(left[index], right[index], f"{path}[{index}]")
            )
        for index in range(common, len(left)):
            differences.append(
                {"path": f"{path}[{index}]", "kind": "removed", "left": left[index]}
            )
        for index in range(common, len(right)):
            differences.append(
                {"path": f"{path}[{index}]", "kind": "added", "right": right[index]}
            )
        return differences

    if left != right:
        differences.append(
            {"path": path, "kind": "value_changed", "left": left, "right": right}
        )
    return differences


## Validation fixtures

In [34]:
import linecache
import textwrap
import types


def module_from_source(source: str, name: str, extra_globals=None):
    """Compile source under a linecache-backed filename for inspect.getsource."""
    source = textwrap.dedent(source).lstrip("\n")
    if not source.endswith("\n"):
        source += "\n"
    filename = f"<{name}>"
    lines = source.splitlines(keepends=True)
    linecache.cache[filename] = (len(source), None, lines, filename)
    module = types.ModuleType(name)
    module.__file__ = filename
    if extra_globals:
        module.__dict__.update(extra_globals)
    exec(compile(source, filename, "exec"), module.__dict__)
    return module


def node(tree: EnvTree, *path):
    """Get an already-created node without changing the analyzed tree."""
    return tree.get_node(path, create=False)


cases = module_from_source(
    r'''
    import copy
    from copy import deepcopy

    def rule1(env):
        pending = env["branch"]["leaf"]
        env["branch"]
        env["other"]["leaf"]

    def rule2(env):
        known = env["known"]["leaf"]
        full = copy.deepcopy(env)
        source = env["source"]
        partial = deepcopy(source)
        full["full_leaf"]
        partial["partial_leaf"]

    def rule3(env):
        env2 = env
        env3 = env2
        branch = env3["a"]
        deep = branch["b"]
        deep["leaf"]
        deep = object()
        deep["ignored"]
        env2 = copy.deepcopy(env)
        env2["copy_leaf"]
        env3 = None
        env3["ignored_root"]

    def rule4(env):
        known = env["branch"]["leaf"]["known"]
        env["branch"]["leaf"] = 0
        env["branch"]["leaf"]["late"]
        env["branch"]["sibling"]
        env["branch"]
        env["target"] = env["rhs"]

    def rule5(env):
        untouched = env["not_overwritten"]
        env["before"]["x"] = 1
        first(env)
        env["after"]["y"] = 2
        clone = copy.deepcopy(env)
        clone["copy_only"] = 3
        service.second(payload=clone)
        consume(env["before"])

    def rule6(env):
        alias_only = env["alias_only"]
        env["overwrite_only"] = 1
        send(env)
        env["used"]["leaf"]

    def consistent_if_elif(env, mode):
        if mode == 0:
            env["same"]
            forward(env)
        elif mode == 1:
            env["same"]
            forward(env)
        else:
            env["same"]
            forward(env)

    def divergent_if(env, flag):
        if flag:
            env["left"]
        else:
            env["right"]

    def converges_after_if(env, flag):
        if flag:
            env["common"]
        env["common"]

    def consistent_match(env, value):
        match value:
            case 1:
                env["same_match"]
            case _:
                env["same_match"]

    def divergent_match(env, value):
        match value:
            case 1:
                env["match_left"]
            case _:
                env["match_right"]

    def nonexhaustive_match(env, value):
        match value:
            case 1:
                env["only_if_one"]

    def consistent_try(env):
        try:
            env["same_try"]
            forward(env)
        except ValueError:
            env["same_try"]
            forward(env)

    def divergent_try(env):
        try:
            env["try_side"]
        except ValueError:
            env["except_side"]

    def dynamic_key(env, key):
        env[key]

    def assignment_evaluation_order(env):
        alias = env["value"]
        alias = alias + 1
    ''',
    "env_schema_validation_cases",
)

validation_results = []


def passed(label: str):
    validation_results.append(label)
    print("✓", label)


## Rule 1 — tree nodes and `used` propagation

In [35]:
# Rule 1: parent/child tree structure, literal keys, and recursive use marking.
trees, calls = analyze_env_schema(cases.rule1)
root = trees[0].root

assert root.children["branch"].used
assert root.children["branch"].children["leaf"].used
assert not root.children["other"].used
assert root.children["other"].children["leaf"].used
assert calls == []

passed("Rule 1 — tree structure and used flags")
print_analysis(trees, calls)


✓ Rule 1 — tree structure and used flags
{
  "trees": [
    {
      "tree_id": 0,
      "copied_from": null,
      "reduced_to_overwrites": false,
      "root": {
        "key": {
          "type": "str",
          "repr": "'<root:env>'"
        },
        "used": false,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'branch'"
            },
            "used": true,
            "overwritten": false,
            "children": [
              {
                "key": {
                  "type": "str",
                  "repr": "'leaf'"
                },
                "used": true,
                "overwritten": false,
                "children": []
              }
            ]
          },
          {
            "key": {
              "type": "str",
              "repr": "'other'"
            },
            "used": false,
            "overwritten": false,
            "children": [
              

## Rules 2–3 — deep copies and active aliases

In [36]:
# Rule 2: every deepcopy becomes an independent EnvTree with origin metadata.
trees, calls = analyze_env_schema(cases.rule2)
assert len(trees) == 3
assert trees[1].copied_from == CopyOrigin(0, ())
assert trees[2].copied_from == CopyOrigin(0, ("source",))
assert node(trees[1], "known", "leaf").used is False
assert node(trees[1], "full_leaf").used is True
assert node(trees[2], "partial_leaf").used is True
assert [call.function_name for call in calls] == ["copy.deepcopy"]
passed("Rule 2 — independently tracked deep copies")

# Rule 3: aliases can be chained to arbitrary depth and are removed on rebinding.
trees, calls = analyze_env_schema(cases.rule3)
assert node(trees[0], "a", "b", "leaf").used
assert "ignored" not in node(trees[0], "a", "b").children
assert "ignored_root" not in trees[0].root.children
assert len(trees) == 2
assert node(trees[1], "copy_leaf").used
passed("Rule 3 — arbitrary-depth aliases and alias override")


✓ Rule 2 — independently tracked deep copies
✓ Rule 3 — arbitrary-depth aliases and alias override


In [37]:
for tree in trees:
    print(tree.get_used_leaves())

{'a.b.leaf'}
{'a.b.leaf', 'copy_leaf'}


## Rule 4 — overwrite state

In [38]:
# Rule 4: overwrites propagate to existing descendants; new descendants inherit
# overwritten=True; reads of overwritten nodes do not alter used.
trees, calls = analyze_env_schema(cases.rule4)
branch = node(trees[0], "branch")
leaf = node(trees[0], "branch", "leaf")

assert branch.used and node(trees[0], "branch", "sibling").used
assert leaf.overwritten and not leaf.used
assert node(trees[0], "branch", "leaf", "known").overwritten
assert not node(trees[0], "branch", "leaf", "known").used
assert node(trees[0], "branch", "leaf", "late").overwritten
assert not node(trees[0], "branch", "leaf", "late").used
assert node(trees[0], "target").overwritten
assert not node(trees[0], "target").used
assert node(trees[0], "rhs").used

passed("Rule 4 — overwrite propagation and read suppression")


✓ Rule 4 — overwrite propagation and read suppression


## Rule 5 — `FunctionCall` snapshots

In [39]:
# Rule 5: direct root arguments create independent reduced snapshots.
trees, calls = analyze_env_schema(cases.rule5)
print(calls)
assert [call.function_name for call in calls] == [
    "first",
    "copy.deepcopy",
    "service.second",
]

first_snapshot = calls[0].overwritten_tree
copy_snapshot = calls[1].overwritten_tree
service_snapshot = calls[2].overwritten_tree

# The first snapshot cannot change when later overwrites occur.
assert node(first_snapshot, "before", "x").overwritten
assert not node(first_snapshot, "before").used
assert "after" not in first_snapshot.root.children
assert "not_overwritten" not in first_snapshot.root.children

# The deepcopy call sees both original overwrites at its own call time.
assert node(copy_snapshot, "before", "x").overwritten
assert node(copy_snapshot, "after", "y").overwritten

# The copied root later contains inherited overwrites plus its own overwrite.
assert node(service_snapshot, "before", "x").overwritten
assert node(service_snapshot, "after", "y").overwritten
assert node(service_snapshot, "copy_only").overwritten

# Passing env["before"] is a branch use, not a root FunctionCall.
assert node(trees[0], "before").used
assert not node(trees[0], "before", "x").used

passed("Rule 5 — root-call snapshots and reduced overwrite trees")


[FunctionCall(function_name='first', overwritten_tree={
  "tree_id": 0,
  "copied_from": null,
  "reduced_to_overwrites": true,
  "root": {
    "key": {
      "type": "str",
      "repr": "'<root:env>'"
    },
    "used": false,
    "overwritten": false,
    "children": [
      {
        "key": {
          "type": "str",
          "repr": "'before'"
        },
        "used": false,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'x'"
            },
            "used": false,
            "overwritten": true,
            "children": []
          }
        ]
      }
    ]
  }
}), FunctionCall(function_name='copy.deepcopy', overwritten_tree={
  "tree_id": 0,
  "copied_from": null,
  "reduced_to_overwrites": true,
  "root": {
    "key": {
      "type": "str",
      "repr": "'<root:env>'"
    },
    "used": false,
    "overwritten": false,
    "children": [
      {
        "key": {
          "type": "st

In [40]:
for tree in trees:
    print(tree)
    print(tree.get_used_leaves())

{
  "tree_id": 0,
  "copied_from": null,
  "reduced_to_overwrites": false,
  "root": {
    "key": {
      "type": "str",
      "repr": "'<root:env>'"
    },
    "used": false,
    "overwritten": false,
    "children": [
      {
        "key": {
          "type": "str",
          "repr": "'after'"
        },
        "used": false,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'y'"
            },
            "used": false,
            "overwritten": true,
            "children": []
          }
        ]
      },
      {
        "key": {
          "type": "str",
          "repr": "'before'"
        },
        "used": true,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'x'"
            },
            "used": false,
            "overwritten": true,
            "children": []
          }
        ]
      },
      

## Rules 6–7 — usage exclusions and outputs

In [41]:
# Rule 6: alias creation, overwrite targets, and direct root passing are not uses.
trees, calls = analyze_env_schema(cases.rule6)
assert not node(trees[0], "alias_only").used
assert node(trees[0], "overwrite_only").overwritten
assert not node(trees[0], "overwrite_only").used
assert not trees[0].root.used
assert not node(trees[0], "used").used
assert node(trees[0], "used", "leaf").used
assert [call.function_name for call in calls] == ["send"]
passed("Rule 6 — usage exceptions")

# Rule 7: exact requested output container types.
assert isinstance(trees, list)
assert isinstance(calls, list)
assert all(isinstance(tree, EnvTree) for tree in trees)
assert all(isinstance(call, FunctionCall) for call in calls)
passed("Rule 7 — output types")


✓ Rule 6 — usage exceptions
✓ Rule 7 — output types


## Rule 8 — control-flow consistency

In [42]:
# Rule 8: consistent paths pass.
analyze_env_schema(cases.consistent_if_elif)
analyze_env_schema(cases.converges_after_if)
analyze_env_schema(cases.consistent_match)
analyze_env_schema(cases.consistent_try)

# Divergent paths raise and expose branch-specific structural differences.
expected_failures = [
    cases.divergent_if,
    cases.divergent_match,
    cases.nonexhaustive_match,
    cases.divergent_try,
]

captured = {}
logger_was_disabled = logger.disabled
logger.disabled = True  # Avoid four expected ERROR records in notebook output.
try:
    for function in expected_failures:
        try:
            analyze_env_schema(function)
        except DynamicSchemaError as exc:
            assert exc.branch_differences
            assert exc.branch_differences[0]["diff"]
            captured[function.__name__] = exc.branch_differences[0]
        else:
            raise AssertionError(
                f"Expected DynamicSchemaError for {function.__name__}"
            )
finally:
    logger.disabled = logger_was_disabled

passed("Rule 8 — if/elif, match-case, and try-except consistency")
print("Example divergent-if diagnostic:")
print(json.dumps(captured["divergent_if"], indent=2, default=repr))


✓ Rule 8 — if/elif, match-case, and try-except consistency
Example divergent-if diagnostic:
{
  "baseline_paths": [
    "entry -> if@66:body"
  ],
  "compared_paths": [
    "entry -> if@66:else"
  ],
  "diff": {
    "values_changed": {
      "root['trees'][0]['root']['children'][0]['key']['repr']": {
        "new_value": "'right'",
        "old_value": "'left'"
      }
    }
  }
}


## Additional static-analysis checks

In [43]:
# Additional safety: dynamic keys are rejected rather than guessed.
try:
    analyze_env_schema(cases.dynamic_key)
except DynamicKeyError:
    pass
else:
    raise AssertionError("Expected DynamicKeyError")
passed("Additional — dynamic-key rejection")

# Assignment RHS is analyzed before rebinding the target alias.
trees, _ = analyze_env_schema(cases.assignment_evaluation_order)
assert node(trees[0], "value").used
passed("Additional — Python assignment evaluation order")

print(f"\n{len(validation_results)} validation groups passed.")


✓ Additional — dynamic-key rejection
✓ Additional — Python assignment evaluation order

10 validation groups passed.


## Optional `__init_subclass__` integration

In [44]:
class AutoAnalyzed:
    """Minimal integration showing analysis at subclass-definition time."""

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        run = cls.__dict__.get("_run")
        if run is not None:
            cls.env_trees, cls.function_calls = analyze_env_schema(run)

class UserPath(AutoAnalyzed):
    def _run(self, env):
        env["input"]["value"]
        
assert node(UserPath.env_trees[0], "input", "value").used
print("Subclass analyzed during class creation:")
print_analysis(
    UserPath.env_trees,
    UserPath.function_calls,
)
for tree in UserPath.env_trees:
    print(tree.get_used_leaves())

Subclass analyzed during class creation:
{
  "trees": [
    {
      "tree_id": 0,
      "copied_from": null,
      "reduced_to_overwrites": false,
      "root": {
        "key": {
          "type": "str",
          "repr": "'<root:env>'"
        },
        "used": false,
        "overwritten": false,
        "children": [
          {
            "key": {
              "type": "str",
              "repr": "'input'"
            },
            "used": false,
            "overwritten": false,
            "children": [
              {
                "key": {
                  "type": "str",
                  "repr": "'value'"
                },
                "used": true,
                "overwritten": false,
                "children": []
              }
            ]
          }
        ]
      }
    }
  ],
  "function_calls": []
}
{'input.value'}


## Practical boundaries

The implementation is intentionally conservative where Python cannot be resolved statically:

- Keys must be literals. Supporting dynamic keys would require a symbolic-key representation or runtime tracing.
- Deep-copy recognition is name based. Add aliases such as `"cp.deepcopy"` through `deepcopy_names=`.
- A `try` handler is modeled as a logical alternative beginning at try entry. Precise exception-prefix analysis would require exception-effect modeling.
- Loops are additionally supported as zero iterations versus one-or-more iterations. Loop-dependent copies or calls are therefore reported as dynamic.
- Reflection, mutations hidden inside arbitrary functions, and aliases stored inside containers require interprocedural analysis or runtime tracing.

# Compare two functions for equivalence (ignoring formatting)

In [45]:
import ast
import inspect
import textwrap


class FunctionalNormalizer(ast.NodeTransformer):
    """
    Normalize a Python function AST by removing source-level details that
    do not affect its basic computational structure.
    """

    def visit_FunctionDef(self, node):
        # Function name itself does not affect the function body.
        node.name = "<function>"

        # Decorators affect how the function object is constructed, but if
        # we're comparing the function implementation itself, ignore them.
        node.decorator_list = []

        # Remove docstring.
        if (
            node.body
            and isinstance(node.body[0], ast.Expr)
            and isinstance(node.body[0].value, ast.Constant)
            and isinstance(node.body[0].value.value, str)
        ):
            node.body.pop(0)

        self.generic_visit(node)
        return node

    def visit_AsyncFunctionDef(self, node):
        node.name = "<function>"
        node.decorator_list = []

        if (
            node.body
            and isinstance(node.body[0], ast.Expr)
            and isinstance(node.body[0].value, ast.Constant)
            and isinstance(node.body[0].value.value, str)
        ):
            node.body.pop(0)

        self.generic_visit(node)
        return node


def normalized_function_ast(func):
    """
    Return a normalized AST representation of a function.

    Ignores:
        - Whitespace
        - Comments
        - Formatting
        - Function name
        - Docstring
        - Decorators
        - Source locations / line numbers

    Parameters
    ----------
    func : callable
        Function handle to normalize.

    Returns
    -------
    str
        Canonical AST representation of the function.
    """
    source = textwrap.dedent(inspect.getsource(func))
    tree = ast.parse(source)

    tree = FunctionalNormalizer().visit(tree)
    ast.fix_missing_locations(tree)

    return ast.dump(
        tree,
        annotate_fields=True,
        include_attributes=False,
    )


def functions_structurally_equal(func1, func2):
    """
    Determine whether two functions have the same normalized AST.
    """
    return normalized_function_ast(func1) == normalized_function_ast(func2)


# ---------------------------------------------------------------------
# Example
# ---------------------------------------------------------------------

def function_a(x):
    """Square x and add one."""
    # This is a comment.
    y = x ** 2
    return y + 1


def completely_different_name(x):

    y=x**2

    return y+1


def function_c(x):
    y = x ** 3
    return y + 1


print(functions_structurally_equal(function_a, completely_different_name))
# True

print(functions_structurally_equal(function_a, function_c))
# False

print(normalized_function_ast(function_a))

True
False
Module(body=[FunctionDef(name='<function>', args=arguments(posonlyargs=[], args=[arg(arg='x')], kwonlyargs=[], kw_defaults=[], defaults=[]), body=[Assign(targets=[Name(id='y', ctx=Store())], value=BinOp(left=Name(id='x', ctx=Load()), op=Pow(), right=Constant(value=2))), Return(value=BinOp(left=Name(id='y', ctx=Load()), op=Add(), right=Constant(value=1)))], decorator_list=[], type_params=[])], type_ignores=[])


In [46]:
import hashlib
signature_a = hashlib.sha256(
    normalized_function_ast(function_a).encode()
).hexdigest()
signature_b = hashlib.sha256(
    normalized_function_ast(completely_different_name).encode()
).hexdigest()
print(signature_a)
print(signature_b)
assert signature_a == signature_b

e069e7268bcc11bab2f1a13a337960597f8160d5610b0734ad551f2616da57cc
e069e7268bcc11bab2f1a13a337960597f8160d5610b0734ad551f2616da57cc


# Function Decorator

In [53]:
import inspect
from functools import update_wrapper
from typing import Callable, Generic, ParamSpec, TypeVar, Dict


P = ParamSpec("P")
R = TypeVar("R")

jpath_registry = {}

class JPath(Generic[P, R]):
    """
    A callable wrapper that preserves parameter and return type information.
    
    This class wraps a callable function while maintaining its original signature
    for static type checking and runtime introspection.
    
    Parameters
    ----------
    func : Callable[P, R]
        The function to wrap. Must have exactly three parameters:
        'env', 'partial_result', and 'path_options', in that order.
    
    Attributes
    ----------
    func : Callable[P, R]
        The wrapped function.
    __signature__ : inspect.Signature
        The signature of the original function, used by inspect.signature().
    
    Raises
    ------
    TypeError
        If the provided function does not have exactly the required parameters
        ('env', 'partial_result', 'path_options') in the correct order.
    
    Examples
    --------
    >>> @jmap
    ... def my_path(env: int, partial_result: str, path_options: bool) -> int:
    ...     return env
    
    >>> result = my_path(1, "test", True)
    >>> result
    1
    """
    
    REQUIRED_ARGS = ("env", "partial_result", "path_options")
    
    def __init__(self, func: Callable[P, R]) -> None:
        """
        Initialize the JPath wrapper.
        
        Parameters
        ----------
        func : Callable[P, R]
            The function to wrap.
        
        Raises
        ------
        TypeError
            If the function signature does not match requirements.
        """
        signature = inspect.signature(func)
        actual_args = tuple(signature.parameters.keys())
        
        if actual_args != self.REQUIRED_ARGS:
            raise TypeError(
                f"@jmap requires arguments {self.REQUIRED_ARGS}, in that order. "
                f"{func.__qualname__} has arguments {actual_args}."
            )
        
        self.func = func
        self.__signature__ = signature
        update_wrapper(self, func)
        
        ################################# Analyze Schema ########################################
        self.env_trees, self.function_calls = analyze_env_schema(func)
        
        # Register the path in the global path registry
        jpath_registry[func.__module__ + "." + func.__qualname__] = self
    
    def __call__(self, *args: P.args, **kwargs: P.kwargs) -> R:
        """
        Call the wrapped function.
        
        Parameters
        ----------
        *args : P.args
            Positional arguments to pass to the wrapped function.
        **kwargs : P.kwargs
            Keyword arguments to pass to the wrapped function.
        
        Returns
        -------
        R
            The return value from the wrapped function.
        """
        return self.func(*args, **kwargs)
    
    def evaluate_schema(self, env: dict[str, Any]) -> Dict:
        """
        Evaluate the schema of the wrapped function. 
        
        Returns
        -------
        set[str]
            A set of strings representing the key paths, delimited by ".", in the evaluated schema.
        """
        env_schema = {}
        def add_path_to_schema(path: str) -> None:
            keys = path.split(".")
            current = env_schema
            current_env = env
            for key in keys[:-1]:
                if key not in current:
                    current[key] = {}
                current = current[key]
                current_env = current_env.get(key)
            current[keys[-1]] = type(current_env[keys[-1]])

        used_leaves = self.get_used_leaves(env=env)
        for path in used_leaves:
            add_path_to_schema(path)
        return env_schema

    def get_used_leaves(self, env: dict[str, Any] | None = None) -> set[str]:
        """
        Extract and return the schema of variables utilized in the user defined func
        
        Returns
        -------
        set[str]
            A set containing unique usage entries from all environment usage in func.
        """
        used_leaves = set()
        for tree in self.env_trees:
            used_leaves = used_leaves | tree.get_used_leaves(env=env)
        for fn_call in self.function_calls:
            qualified_name = fn_call.get_runtime_name()
            if qualified_name in jpath_registry:
                if env is not None:
                    used_leaves |= jpath_registry[qualified_name].get_used_leaves(env=env) - fn_call.get_overwritten_leaves(env=env)
                else:
                    untrimmed_fn_usage = jpath_registry[qualified_name].get_used_leaves(env=env)
                    overwritten_leaves = fn_call.get_overwritten_leaves(env=env)
                    trimmed_fn_usage = set()
                    for path in untrimmed_fn_usage:
                        keys = path.split(".")
                        overwritten = False
                        for i in range(len(keys)):
                            if ".".join(keys[:i+1]) in overwritten_leaves:
                                overwritten = True
                                break
                        if not overwritten:
                            trimmed_fn_usage.add(path)
                    used_leaves |= trimmed_fn_usage
        return used_leaves
    
    def migrate_version(
        self,
        update_fn: Callable,
        source_version: int | None = None,
        target_version: int | None = None,
    ) -> None:
        """
        Perform a migration of the JPath version.
        
        Parameters
        ----------
        update_fn : Callable
            A function that performs the version update operation.
        source_version : int, optional
            The source version to migrate from. Default is None.
        target_version : int, optional
            The target version to migrate to. Default is None.
        
        Returns
        -------
        None
        """
        pass


def jmap(func: Callable[P, R]) -> JPath[P, R]:
    """
    Decorator to convert a function into a JPath wrapper.
    
    This decorator preserves the original function's call signature for both
    static type checking and runtime introspection.
    
    Parameters
    ----------
    func : Callable[P, R]
        The function to wrap. Must have exactly three parameters:
        'env', 'partial_result', and 'path_options', in that order.
    
    Returns
    -------
    JPath[P, R]
        A JPath instance wrapping the original function.
    
    Examples
    --------
    >>> @jmap
    ... def my_path(env: int, partial_result: str, path_options: bool) -> int:
    ...     '''Test comments.'''
    ...     return env
    """
    return JPath(func)


@jmap
def my_path(env, partial_result, path_options):
    """Test comments for MY PATH!!!"""
    print("Hello world")
    env["b"]["1"]
    a = env["a"]
    two = env["b"]["2"]

@jmap
def my_super_path(env, partial_result, path_options):
    """Test comments for MY SUPER PATH!!!"""
    print("Hello world")
    my_path(env, partial_result, path_options)
    env["b"]["2"]

env = {
    "a" : "a",
    "b" : {
        "1" : 1,
        "2" : 2
    }
}
@jmap
def my_super_overwritten_path(env, partial_result, path_options):
    """Test comments for MY SUPER OVERWRITTEN PATH!!!"""
    print("Hello world")
    env["b"] = {
        "1" : 1,
        "2" : 2
    }
    my_path(env, partial_result, path_options)
    env["b"]["2"]

env = {
    "a" : "a",
    "b" : {
        "1" : 1,
        "2" : 2
    }
}

# The IDE knows the original call signature.
result = my_path(env, "test", True)

try:
    @jmap
    def my_path_error(env, partial_result):
        pass
except TypeError as e:
    print(f"Caught expected TypeError: {e}")

# The IDE also knows that this is a JPath:
print(f"Schema: {my_path.get_used_leaves()}")

# And it knows JPath attributes:
original_function = my_path.func

# Runtime introspection also works.
print(inspect.signature(my_path))
# (env: int, partial_result: str, path_options: bool) -> int
print(f"Path Registry: {jpath_registry}")
print("######################################################################")
print(f"Super Usage: {my_super_path.get_used_leaves()}")
print(f"Super Function Call Overwrites: {[f"{call.function_name}: {call.overwritten_tree.get_overwritten_leaves()}" for call in my_super_path.function_calls]}")
print(f"Super Env Schema: {my_super_path.evaluate_schema(env)}")
print("######################################################################")
print(f"Super Overwritten Usage: {my_super_overwritten_path.get_used_leaves()}")
print(f"Super Function Call Overwrites: {[f"{call.function_name}: {call.overwritten_tree.get_overwritten_leaves()}" for call in my_super_overwritten_path.function_calls]}")
print(f"Super Overwritten Env Schema: {my_super_overwritten_path.evaluate_schema(env)}")


Hello world
Caught expected TypeError: @jmap requires arguments ('env', 'partial_result', 'path_options'), in that order. my_path_error has arguments ('env', 'partial_result').
Schema: {'b.1'}
(env, partial_result, path_options)
Path Registry: {'__main__.my_path': <__main__.JPath object at 0x7f884971c680>, '__main__.my_super_path': <__main__.JPath object at 0x7f882f3ca810>, '__main__.my_super_overwritten_path': <__main__.JPath object at 0x7f88221d3a40>}
######################################################################
Super Usage: {'b.2', 'b.1'}
Super Function Call Overwrites: ['my_path: set()']
Super Env Schema: {'b': {'2': <class 'int'>, '1': <class 'int'>}}
######################################################################
Super Overwritten Usage: set()
Super Function Call Overwrites: ["my_path: {'b'}"]
Super Overwritten Env Schema: {}


In [48]:
import scipy

cons = scipy.interpolate
test_container = type("TestContainer", (), {})()
test_container.my_path = my_path
calls = ["cons.make_interp_spline", "scipy.interpolate.make_interp_spline", "my_path", "test_container.my_path"]
print(cons.__name__)

for call in calls:
    print(get_qualified_name(call))

scipy.interpolate
scipy.interpolate._bsplines.make_interp_spline
scipy.interpolate._bsplines.make_interp_spline
__main__.my_path
__main__.my_path
